<a href="https://colab.research.google.com/github/ERA-Software/computational-data-analysis/blob/main/notebooks/T5_improved_regression_with_engineered_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏗️ Structural Materials Hackathon
## Predicting Concrete Compressive Strength for Reliability Assessment

---

| | |
|---|---|
| ⏱️ **Duration** | 60 minutes |
| 🎯 **Goal** | Minimise **RMSE** [MPa] of your regression model on the held-out test set |
| 📦 **Dataset** | UCI Concrete Compressive Strength — Yeh (1998) |
| 📐 **Domain** | Structural reliability / material science |
| 🏆 **Scoring** | Lower RMSE wins — leaderboard at the end of the session |

---

> *"The real competition is not in choosing the right model — it is in understanding your data and the physics behind it."*

---

## 📚 Part 1 — Introduction to Regression

### What is Regression?

**Regression** is a supervised learning task where we learn a mapping $f: \mathbf{x} \rightarrow y$ from input features to a **continuous** target variable.

| Task | Target | Example |
|------|--------|---------|
| **Regression** | Continuous | Concrete strength [MPa], beam deflection [mm] |
| Classification | Categorical | Safe / Unsafe, Pass / Fail |

### Why Does It Matter for Structural Reliability?

In structural reliability the **limit state function** separates safe from unsafe states:

$$g(\mathbf{x}) = R(\mathbf{x}) - S(\mathbf{x}) \quad\begin{cases} > 0 & \text{safe} \\ = 0 & \text{limit state surface} \\ < 0 & \text{failure} \end{cases}$$

The **resistance** $R$ of a reinforced concrete beam in bending is:

$$M_R = A_s f_y \left(d - \frac{A_s f_y}{1.7\, f_c\, b}\right)$$

$f_c$ appears in the denominator — an error of a few MPa in predicting concrete strength propagates directly into the capacity estimate and therefore into the probability of failure. A regression surrogate that predicts $f_c$ from mix design avoids costly lab testing and enables fast Monte Carlo reliability analyses over thousands of mix configurations.

### The Standard ML Pipeline

```
Raw Mix Data
      │
      ▼
 EDA & Visualisation
      │
      ▼
 Feature Engineering  ←── Physics knowledge goes here!
      │
      ▼
 Preprocessing (imputation, scaling)
      │
      ▼
 Train / Validation / Test Split
      │
      ▼
 Model + Hyperparameter Tuning (GridSearchCV)
      │
      ▼
 Evaluate on Test Set → RMSE
```

### Regression Metrics

| Metric | Formula | Notes |
|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i-\hat{y}_i)^2}$ | Same unit as target (MPa); penalises large errors |
| **MAE** | $\frac{1}{n}\sum|y_i-\hat{y}_i|$ | Robust to outliers |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Fraction of variance explained (1 = perfect) |

**Lower RMSE = better model. 🏆**

---
## 🔧 Part 2 — Scikit-learn Crash Course

### Resources

| Resource | URL |
|----------|-----|
| Official docs | https://scikit-learn.org/stable/ |
| User Guide | https://scikit-learn.org/stable/user_guide.html |
| API Reference | https://scikit-learn.org/stable/modules/classes.html |
| Prompt engineering guide | https://docs.claude.com/en/docs/build-with-claude/prompt-engineering/overview |

### 2.1 — The Universal API

In [ ]:
# Every scikit-learn estimator follows the same three-step pattern
from sklearn.linear_model import Ridge
import numpy as np

X_demo = np.random.randn(100, 4)
y_demo = np.random.randn(100)

model = Ridge(alpha=1.0)         # 1. Instantiate with hyperparameters
model.fit(X_demo, y_demo)        # 2. Train on training data
y_pred = model.predict(X_demo)   # 3. Predict on new data

print("Coefficients:", model.coef_.round(3))

### 2.2 — Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X_demo, y_demo,
    test_size=0.2,    # 20% held out
    random_state=42   # ⚠️ NEVER change this in the hackathon!
)
print(f"Train: {X_tr.shape[0]} samples | Test: {X_te.shape[0]} samples")

### 2.3 — Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer

# ── Impute missing values ─────────────────────────────────────────────────────
imputer  = SimpleImputer(strategy='median')    # or 'mean', 'most_frequent'
X_imp    = imputer.fit_transform(X_tr)         # fit on TRAIN only
X_imp_te = imputer.transform(X_te)             # apply same transform to test

# ── Scale features ────────────────────────────────────────────────────────────
# StandardScaler: zero mean, unit variance  →  sensitive to outliers
# RobustScaler:   uses median / IQR         →  robust to outliers ✅
scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X_imp)         # NEVER fit on test set!

# ⚠️  KEY RULE: fit_transform on TRAIN, transform-only on TEST
#     Fitting on test data = data leakage = invalid results
print("Scaled train median ≈ 0:", X_scaled.mean(axis=0).round(2))

### 2.4 — Regression Models

In [ ]:
from sklearn.linear_model import (
    LinearRegression,   # OLS — no regularisation
    Ridge,              # L2: shrinks all coefficients
    Lasso,              # L1: zeros out irrelevant features (feature selection)
    ElasticNet          # L1 + L2 combined
)

# Ridge — good when many features are mildly correlated
ridge = Ridge(alpha=1.0)     # alpha=0 → plain OLS; alpha→∞ → all coefs→0

# Lasso — forces some coefficients to exactly zero
lasso = Lasso(alpha=0.1, max_iter=10000)

# ElasticNet — combines both penalties
enet = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
# l1_ratio=0 → Ridge; l1_ratio=1 → Lasso

print("Models defined ✅")

### 2.5 — Pipelines (Best Practice)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Pipeline chains all steps and handles train/test correctly — no leakage
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
    ('model',   Ridge(alpha=1.0))
])

pipe.fit(X_tr, y_tr)
rmse_demo = np.sqrt(mean_squared_error(y_te, pipe.predict(X_te)))
print(f"Demo Pipeline RMSE: {rmse_demo:.4f}")

### 2.6 — Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

# Parameter names follow the pattern:  step_name__parameter_name
param_grid = {
    'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

gs = GridSearchCV(
    estimator=Pipeline([
        ('imp',   SimpleImputer(strategy='median')),
        ('sc',    RobustScaler()),
        ('model', Ridge())
    ]),
    param_grid=param_grid,
    cv=5,                                    # 5-fold cross-validation
    scoring='neg_root_mean_squared_error',   # maximise -RMSE = minimise RMSE
    n_jobs=-1                                # use all CPU cores
)

gs.fit(X_tr, y_tr)
print("Best alpha  :", gs.best_params_)
print("Best CV RMSE:", round(-gs.best_score_, 4))

---
## 🏗️ Part 3 — The Hackathon Challenge

### Problem Statement: Concrete Compressive Strength as Structural Resistance

Concrete compressive strength $f_c$ [MPa] is the fundamental material parameter controlling structural capacity. It is **highly nonlinear** in its ingredients and curing age.

Rather than running expensive laboratory tests on every mix design, your task is to build a **regression surrogate model** that predicts $f_c$ from mix proportions — enabling fast probabilistic reliability assessments.

### Dataset: UCI Concrete Compressive Strength (Yeh, 1998)

> Yeh, I.C. (1998). *Modeling of strength of high performance concrete using artificial neural networks.* Cement and Concrete Research, 28(12), 1797–1808.

**1,030 real concrete mix design samples** from laboratory experiments.

| Feature | Symbol | Unit | Physical Role |
|---------|--------|------|---------------|
| `Cement` | $c$ | kg/m³ | Primary binder — main strength driver |
| `BlastFurnaceSlag` | — | kg/m³ | Supplementary binder (steel by-product) |
| `FlyAsh` | — | kg/m³ | Supplementary binder (coal combustion) |
| `Water` | $w$ | kg/m³ | Controls w/c ratio — inversely related to strength |
| `Superplasticizer` | SP | kg/m³ | Chemical additive improving workability |
| `CoarseAggregate` | — | kg/m³ | Structural filler |
| `FineAggregate` | — | kg/m³ | Structural filler |
| `Age` | $t$ | days | Curing time — strength grows with age |

**Target:** `CompressiveStrength` — concrete compressive strength [MPa]

### ⚠️ Real-World Data Challenges

This is a **real dataset** — the challenges are genuine properties of the data, not injected artificially:

1. **Zero-inflated features** — many mixes contain no `BlastFurnaceSlag` or `FlyAsh` at all
2. **Skewed distributions** — `Age` is heavily right-skewed (many 28-day tests); `Cement` is log-normal
3. **Nonlinear physics** — strength follows Abrams' Law ($f_c \propto A/B^{w/c}$), not a linear function
4. **Feature interactions** — `Water/Cement` is far more predictive than `Water` and `Cement` separately
5. **Correlated features** — binder components are correlated; regularisation is needed

> **⚠️ The rule:** All teams must use **linear regression family only** (LinearRegression, Ridge, Lasso, ElasticNet). The winner is the team with the best **data pipeline**, not the fanciest model.

## 🧠 Core Insight

Concrete compressive strength is governed by two laws from the physical literature:

**1. Abrams' Law (1919)**
$$f_c = \frac{A}{B^{w/c}}$$
The water-to-cement ratio $w/c$ is the dominant strength-governing parameter. `Water` and `Cement` as raw features only let the model fit a hyperplane — they cannot recover the ratio relationship without the interaction term `Water/Cement` being made explicit.

**2. Logarithmic Maturity (Powers, 1949)**
$$f_c(t) \propto f_{c,28} \cdot \frac{\log(1+t)}{\log(29)}$$
A linear model on raw `Age` fits a straight line through a logarithmic curve, producing large systematic residuals at early ($t<7$d) and late ($t>90$d) ages. `log_Age` collapses this to a near-linear relationship.

---
## 🚀 Part 4 — Your Workspace

### Step 0 — Install & Import

In [ ]:
# Install the UCI repository loader if needed
# !pip install ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
print("✅ Imports ready")

### Step 1 — Load the Real Dataset

In [ ]:
!pip install ucimlrepo

In [ ]:
# ── Load the UCI Concrete Compressive Strength dataset ───────────────────────
#
# If ucimlrepo is not installed, run:  pip install ucimlrepo
# Dataset page: https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength
# Citation: Yeh, I.C. (1998). Cement and Concrete Research, 28(12), 1797-1808.
# License: CC BY 4.0

try:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=165)
    df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
    df.columns = [
        'Cement', 'BlastFurnaceSlag', 'FlyAsh', 'Water',
        'Superplasticizer', 'CoarseAggregate', 'FineAggregate',
        'Age', 'CompressiveStrength'
    ]
    print(f"✅ UCI dataset loaded: {df.shape[0]} samples, {df.shape[1]-1} features")
    print(f"   Source: UCI ML Repository — Concrete Compressive Strength (id=165)")

except ImportError:
    raise ImportError(
        "\n\n"
        "  ucimlrepo is not installed.\n"
        "  Run the following cell to install it, then re-run this cell:\n\n"
        "     !pip install ucimlrepo\n"
    )
except Exception as e:
    raise RuntimeError(
        f"\n\n"
        f"  Could not fetch the dataset (network error?): {e}\n\n"
        f"  Alternative: download the Excel file manually from:\n"
        f"  https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength\n"
        f"  then load it with:  df = pd.read_excel('Concrete_Data.xls')\n"
        f"  and rename the columns as shown in the docstring above.\n"
    )

df.head()

### Step 2 — Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print(f"Shape: {df.shape}")
df.describe().round(2)

In [ ]:
# Missing values
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values in the original UCI dataset.")
    print("Note: the real challenge is nonlinearity and feature interactions, not imputation.")
else:
    print(pd.DataFrame({'Missing': missing, 'Pct (%)': (missing/len(df)*100).round(2)})[missing > 0])

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    axes[i].hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Count')
plt.suptitle('Feature Distributions — Note Zero-Inflation in BFS & FlyAsh, Skewed Age',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix\n(What raw correlations tell you — and what they miss)',
          fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print("Raw correlations with CompressiveStrength:")
print(df.corr()['CompressiveStrength'].drop('CompressiveStrength').sort_values().round(3))

In [ ]:
# Target variable analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(df['CompressiveStrength'], bins=45, color='coral', edgecolor='white', alpha=0.85)
ax1.axvline(20, color='red',    linestyle='--', linewidth=2, label='20 MPa — structural min (C20)')
ax1.axvline(40, color='orange', linestyle='--', linewidth=2, label='40 MPa — high strength (C40)')
ax1.set_xlabel('$f_c$ [MPa]'); ax1.set_ylabel('Count')
ax1.set_title('Distribution of Compressive Strength', fontweight='bold'); ax1.legend(fontsize=8)

ax2.scatter(df['Age'], df['CompressiveStrength'], alpha=0.3, s=12, color='steelblue')
ax2.set_xlabel('Age [days]'); ax2.set_ylabel('$f_c$ [MPa]')
ax2.set_title('Strength vs. Age\n(Note the nonlinear / logarithmic shape)', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Samples below 20 MPa (structural minimum): {(df['CompressiveStrength']<20).mean()*100:.1f}%")

**PCA**

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=3)
df_pca = pd.DataFrame(pca.fit_transform(df.drop('CompressiveStrength', axis=1)), columns=['PC1', 'PC2','PC3'])

pca.components_
pca.explained_variance_

In [ ]:
import plotly.express as px

fig1 = px.scatter(df_pca, x='PC1', y='PC2',
                    color=df['CompressiveStrength'],
                    title='2D PCA of Concrete Mix Design, Colored by Compressive Strength')
fig1.show()

fig2 = px.scatter(df_pca, x='PC1', y='PC3',
                    color=df['CompressiveStrength'],
                    title='2D PCA of Concrete Mix Design, Colored by Compressive Strength')
fig2.show()

fig3 = px.scatter(df_pca, x='PC2', y='PC3',
                    color=df['CompressiveStrength'],
                    title='2D PCA of Concrete Mix Design, Colored by Compressive Strength')
fig3.show()


In [ ]:
pca_with_strength = pd.concat([df_pca, df['CompressiveStrength']], axis=1)
correlation_matrix = pca_with_strength.corr()
display(correlation_matrix[['CompressiveStrength']])

### Step 3 — Train / Test Split ⚠️ Do NOT change

In [ ]:
feature_cols = [c for c in df.columns if c != 'CompressiveStrength']
X = df[feature_cols]
y = df['CompressiveStrength']

# ⚠️ Fixed split — do NOT change random_state or test_size (ensures fair leaderboard)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train : {X_train.shape[0]} samples")
print(f"Test  : {X_test.shape[0]} samples  ← held out until final evaluation")

---
## 🛠️ Step 4 — Build Your Pipeline

### 💡 Hints

- **Abrams' Law** (1919): $f_c \propto A / B^{w/c}$ — the water-to-cement ratio `Water/Cement` is the most physically predictive feature ever discovered for concrete strength.
- **Log-transform Age**: strength grows logarithmically with curing time. `np.log1p(Age)` linearises this relationship.
- **Total binder**: many mixes use BFS or FlyAsh as partial cement replacements. `Cement + BFS + FlyAsh` gives the effective binder content.
- **Zero-inflation**: BFS and FlyAsh are exactly 0 in many rows — consider what this means for ratios.
- **RobustScaler**: the real UCI data has no injected outliers, but the skewed distributions make `RobustScaler` (median/IQR) more stable than `StandardScaler` (mean/std).
- **Regularisation**: some engineered features will be correlated. Tune `alpha` in Ridge/Lasso/ElasticNet via `GridSearchCV` with 5-fold CV.

In [ ]:
# ============================================================
#  YOUR CODE STARTS HERE
# ============================================================

def engineer_features(X: pd.DataFrame, q99_water: float = None):
    """
    Add physics-informed features to the concrete mix dataset.

    Parameters
    ----------
    X          : raw feature DataFrame (columns as loaded from UCI)
    q99_water  : 99th percentile of Water computed on the TRAINING set.
                 Pass None to compute from X (training only!).
                 Always pass this value when calling on the test set.

    Returns
    -------
    X_eng      : enriched DataFrame
    q99_water  : the threshold used (store and pass to test set)
    """
    X = X.copy()

    # TODO: Add your engineered features below.
    # Start simple, then add more. Each new feature should have a physical
    # justification — can you explain why it should help?

    # Example 1 — Abrams' Law:
    # X['WC_Ratio'] = X['Water'] / (X['Cement'] + 1e-6)

    # Example 2 — Log Age:
    # X['log_Age'] = np.log1p(X['Age'])

    # Example 3 — Total binder:
    # X['TotalBinder'] = X['Cement'] + X['BlastFurnaceSlag'] + X['FlyAsh']
    # X['WB_Ratio']    = X['Water'] / (X['TotalBinder'] + 1e-6)

    return X, q99_water


# ⚠️ Always compute thresholds from train, apply to both train and test
X_train_eng, q99_water = engineer_features(X_train)
X_test_eng,  _         = engineer_features(X_test, q99_water=q99_water)

print("Features after engineering:", X_train_eng.columns.tolist())

In [ ]:
# ----------------------------
# Build and tune your pipeline
# ----------------------------

my_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # handles any NaNs
    ('scaler',  RobustScaler()),                     # robust to skewed tails
    ('model',   Ridge(alpha=1.0))                    # TODO: try ElasticNet, Lasso
])

# TODO: extend this grid and change the model above to match
param_grid = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    # 'model__l1_ratio': [0.2, 0.5, 0.8],   # uncomment for ElasticNet
}

grid_search = GridSearchCV(
    estimator=my_pipe,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_eng, y_train)

print("\n✅ Best parameters:", grid_search.best_params_)
print("   Best CV RMSE   :", round(-grid_search.best_score_, 4), "MPa")

---
### Step 5 — 🏆 Final Evaluation ⚠️ Do NOT Modify

In [ ]:
# ============================================================
#  FINAL EVALUATION — DO NOT MODIFY THIS CELL
# ============================================================

# Baseline: plain OLS with no feature engineering
baseline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LinearRegression())
])
baseline.fit(X_train, y_train)
rmse_base = np.sqrt(mean_squared_error(y_test, baseline.predict(X_test)))

# Your model
best_model  = grid_search.best_estimator_
y_pred      = best_model.predict(X_test_eng)
rmse_final  = np.sqrt(mean_squared_error(y_test, y_pred))
mae_final   = mean_absolute_error(y_test, y_pred)
r2_final    = r2_score(y_test, y_pred)
improvement = (rmse_base - rmse_final) / rmse_base * 100

print("=" * 55)
print("           HACKATHON RESULTS")
print("=" * 55)
print(f"  Baseline RMSE    :  {rmse_base:.3f} MPa")
print(f"  Your RMSE        :  {rmse_final:.3f} MPa  ✅")
print(f"  MAE              :  {mae_final:.3f} MPa")
print(f"  R²               :  {r2_final:.4f}")
print(f"  Improvement      :  {improvement:.1f}% over baseline")
print("=" * 55)

if rmse_final < 5.5:
    print("🏆  OUTSTANDING — Expert-level pipeline!")
elif rmse_final < 6.5:
    print("🥇  EXCELLENT — Strong feature engineering!")
elif rmse_final < 7.5:
    print("🥈  GOOD — Solid pipeline, room to improve.")
elif rmse_final < rmse_base:
    print("🥉  SOLID — Beat the baseline!")
else:
    print("❌  Below baseline — review your feature engineering.")

In [ ]:
# Residual diagnostics
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_test, y_pred, alpha=0.4, s=18, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect fit')
axes[0].set_xlabel('Actual $f_c$ [MPa]'); axes[0].set_ylabel('Predicted $f_c$ [MPa]')
axes[0].set_title(f'Predicted vs. Actual  |  R² = {r2_final:.3f}', fontweight='bold')
axes[0].legend()

axes[1].scatter(y_pred, residuals, alpha=0.4, s=18, color='coral')
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Predicted $f_c$ [MPa]'); axes[1].set_ylabel('Residual [MPa]')
axes[1].set_title(f'Residuals  |  RMSE = {rmse_final:.3f} MPa', fontweight='bold')

axes[2].hist(residuals, bins=35, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[2].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[2].set_xlabel('Residual [MPa]'); axes[2].set_ylabel('Count')
axes[2].set_title(f'Residual Distribution  |  MAE = {mae_final:.3f} MPa', fontweight='bold')

plt.suptitle(f'Model Diagnostics  |  RMSE = {rmse_final:.2f} MPa  |  R² = {r2_final:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📖 Step 6 — Reflection 

Answer all five questions before submitting:

1. **Which single preprocessing or feature engineering step improved your RMSE the most?** Why?

2. **Abrams' Law** states $f_c \propto A/B^{w/c}$. Why is `Water/Cement` more predictive than `Water` and `Cement` as separate features?

3. **Why should `Age` be log-transformed?** Sketch the expected $f_c$ vs. $t$ curve.

4. **What value of `alpha` did GridSearchCV select for your model?** What would happen if `alpha` were set to 0? To 10,000?

5. **Structural reliability link:** if your model overestimates $f_c$ by 5 MPa on average, does the beam appear safer or less safe than it really is? Explain using the $M_R$ formula from Part 1.

---

## 🗺️ Bonus Challenges

| Challenge | Hint |
|-----------|------|
| Water-to-binder ratio | Replace `Water/Cement` with `Water / (C + BFS + FA)` |
| Interaction term | Does `WB_Ratio × log_Age` improve things? |
| Lasso feature selection | Which features get zeroed out? Do they make physical sense? |
| ElasticNet | Tune both `alpha` and `l1_ratio` in the same `GridSearchCV` |
| SP normalisation | Try `Superplasticizer / Water` as an effectiveness proxy |

---

**Good luck! 🏗️**